# Week 04 — Delayed feedback

**Goal.** Model the conversion lag distribution and correct the bias it puts into a model trained on fresh traffic.

**Deliverable.** Naive vs. Chapelle exponential-delay vs. a Weibull survival variant, plus a predicted-vs-actual lag plot.

**Rough shape of the week.** 2h reading (Chapelle) · 6h building · 1h write-up.

---
### Ground rules (they apply every week)

1. **Beat a dumb baseline or it didn't happen.** Logistic regression or the global mean.
   Log the baseline in the same table as the fancy model.
2. **Split by time, never at random.** `split.time_split` — and call
   `split.check_no_leakage` so the assertion, not your memory, enforces it.
3. **Log every run** with `registry.log_result(...)`, including the ones that lost.
   The losing runs are what make the write-up honest.
4. **Write the finding down** in this week's `README.md` while it is fresh.

### Reading

PDFs are in `papers/` next to this notebook — see `papers/README.md`.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=FutureWarning)

%load_ext autoreload
%autoreload 2

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from adslab import data, metrics, plots, split, registry, encoders, calibration

plots.use_style()
pd.set_option("display.width", 140, "display.max_columns", 60)
print("harness ready")

## The bias, stated precisely

Train a model today on clicks from the last 7 days, labelling "converted so far" as 1 and
everything else 0. A click from 6 days ago has had 6 days to convert; a click from 2
hours ago has had 2 hours. The recent clicks are labelled *wrong* — not noisily, but
systematically in one direction — and the model learns that recency predicts
non-conversion.

Chapelle's move: model two things jointly.

- $\Pr(C=1\mid x)$ — will this ever convert?
- $\Pr(D=d\mid C=1, x)$ — given it converts, how long does it take?

with the delay $D$ exponential, $\lambda(x)=\exp(w_d\cdot x)$. An unconverted click at
elapsed time $e$ then contributes $\Pr(C=0) + \Pr(C=1)\Pr(D>e)$ instead of a hard zero.

In [ ]:
df = data.add_attribution_derived(data.load_attribution())
print(f"{len(df):,} rows, {df.timestamp.max()/86400:.1f} days")

sp = split.time_split(df, "timestamp", train_frac=0.7, val_frac=0.1)
split.check_no_leakage(df, sp)
print(sp)

train, val, test = sp.apply(df)

## 1. The lag distribution

`add_attribution_derived` already gives you `conversion_delay_hours`. Plot it on a log
axis. Report: median delay, p90, and **the fraction of conversions arriving after 24
hours** — that last number is the size of the problem.

Then check whether the delay depends on features. If `lambda` is genuinely constant
across campaigns, the fancy model buys you nothing over a global correction, and knowing
that is worth an hour.

In [ ]:
d = df.conversion_delay_hours.dropna()
print(f"n={len(d):,}  p50={d.median():.2f}h  p90={d.quantile(.9):.1f}h  "
      f">24h: {(d > 24).mean():.1%}  >7d: {(d > 168).mean():.1%}")

fig, ax = plt.subplots()
plots.conversion_delay_hist(d, ax=ax)
print(plots.save(fig, 4, "conversion_delay_distribution"))

## 2. Build the biased world

Pick an observation cutoff `T` inside the window. Everything after `T` is unobservable
"future". Relabel training rows as `converted_by_T`, keeping the true 30-day label aside
for evaluation only.

This is the single most important cell of the week: **the true label must never touch
training**, only evaluation. If it leaks, every result this week is meaningless.

In [ ]:
T = df.timestamp.quantile(0.7)

obs = df[df.timestamp < T].copy()
obs["y_observed"] = ((obs.conversion == 1) & (obs.conversion_timestamp < T)).astype(int)
obs["elapsed"] = T - obs.timestamp

print(f"true CVR={obs.conversion.mean():.4%}  observed-by-T CVR={obs.y_observed.mean():.4%}")
print(f"-> {1 - obs.y_observed.sum()/obs.conversion.sum():.1%} of true positives are still invisible at T")

## 3. Model A — naive

Train on `y_observed`. Evaluate against the *true* label. Then break the evaluation down
by how fresh the impression was: bucket test rows by `elapsed` and plot
`calibration_ratio` per bucket. The naive model should be badly biased on the freshest
bucket and fine on the oldest. That plot is the deliverable.

In [ ]:
# TODO: train naive model, then:
# obs["elapsed_bucket"] = pd.qcut(obs.elapsed, 10, labels=False)
# per-bucket calibration_ratio -> plot

## 4. Model B — Chapelle's delayed-feedback model

Two parameter vectors, trained jointly by maximising

$$\log L = \sum_{\text{converted}}\big[\log p(x) + \log\lambda(x) - \lambda(x)d\big]
        + \sum_{\text{not yet}}\log\big[1-p(x) + p(x)e^{-\lambda(x)e}\big]$$

Implement it in PyTorch (autograd handles the gradients; the paper's hand-derived
gradients are for a 2014 LR system). Two heads on shared features: `p` through a sigmoid,
`log_lambda` linear.

Numerical warning: $\lambda$ wants to run to 0 or ∞ on segments with few conversions.
Clamp `log_lambda` to something like [-12, 4] and say so in the write-up.

In [ ]:
import torch, torch.nn as nn

class DelayedFeedbackModel(nn.Module):
    """Two heads: conversion probability p(x), and exponential delay rate lambda(x)."""
    def __init__(self, n_features):
        super().__init__()
        # TODO
        raise NotImplementedError

    def loss(self, x, y_observed, elapsed):
        # TODO: the log-likelihood above
        raise NotImplementedError

## 5. Model C — Weibull

Exponential assumes a constant hazard: a click is as likely to convert in its 100th hour
as its 1st, given it hasn't yet. Look at your own lag histogram — is that true? Almost
certainly not; there is a spike in the first minutes.

Weibull adds a shape parameter $k$: $\Pr(D>d) = e^{-(\lambda d)^k}$. One extra parameter.
Does it actually help, or does it just fit the training lag better without improving the
CVR estimate? Report both.

In [ ]:
# TODO: swap the survival term for Weibull, refit, compare

## 6. Verdict

Three models, one table, evaluated on the true label. Then the plot that tells the story:
calibration ratio by elapsed-time bucket, all three models on one axis. The naive line
should be dramatically wrong on the left and converge on the right.

In [ ]:
# fig, ax = plt.subplots()
# ... one line per model
# print(plots.save(fig, 4, "calibration_by_elapsed_time"))

---
## Log the results

Every model you tried, including the baseline and including the failures. `notes` is the
one sentence you would say out loud about the run — future-you assembles the write-up
from these, so write it now while you still remember why the run mattered.

In [ ]:
# registry.log_result(
#     week=4,
#     model="lightgbm_hashed_2^18",
#     metrics=metrics.evaluate(y_test, p_test),
#     dataset="attribution",
#     params=dict(n_bits=18, num_leaves=63, lr=0.05),
#     notes="beats LR by 0.011 AUC; most of the gain is from cat3 x cat7 interactions",
# )

print(registry.to_markdown(week=4))

---
## Write it up

Open `README.md` in this folder and fill in the three sections. Keep it to a page.

- **What I built** — one paragraph, no code.
- **What the numbers say** — paste the table above; say which comparison is the honest one.
- **What surprised me** — the part worth reading. If nothing surprised you, you probably
  did not stress the model hard enough.

Then commit:

```bash
git add week04_* results/
git commit -m "week 04: <the finding, not the task>"
```